# IMTalker — Emotion LoRA Training (notebook 5/5)

Adds **emotion conditioning** to your pretrained IMTalker generator using targeted LoRA
fine-tuning. After training, you can make the avatar express **neutral, happy, sad,
angry, surprise, or fear** by setting a single integer at inference time — no
architecture surgery on the real-time pipeline required.

### Why Test Videos Previously Showed No Face (Fixed)
In prior runs, only the style LoRA checkpoint (`ditto_blink_lora...`, 92 keys) was specified
as the pretrained weights. Because that checkpoint contains *only* LoRA delta matrices and
**zero base generator weights**, the core 38.8M-parameter generator remained 100% random Gaussian
noise. This caused flow matching to output invalid motion latents, and the renderer produced black frames.
**Fix implemented**:
1. The full pretrained base generator (`generator.ckpt`, 108 base keys) is loaded first.
2. The style LoRA (`ditto_blink_lora...`, 46 layers) is mathematically merged into the base weights: $\Delta W = \frac{\alpha}{r} (B \times A)$.
3. The emotion LoRA adapters and emotion conditioner are attached and trained on top of the fully functional avatar.
4. `TEST_CROP = False` is set by default so the robot face is directly resized to 512×512 without failing 2D landmark face detection.

**What this notebook does**
- Validates the environment (GPU, CUDA, PyTorch 2.x, dependencies)
- **Autonomously downloads all required models & checkpoints** (wav2vec2, renderer, base generator, and style LoRA)
- Verifies dataset structure and label distribution across all 6 emotions (`neutral`, `happy`, `sad`, `angry`, `surprise`, `fear`)
- Launches emotion LoRA training (Option A: multi-GPU background process with DDP; Option B: single GPU direct)
- Automatically renders test video after each epoch with the selected test emotion
- Validates the trained checkpoint and verifies learned emotion vectors


In [ ]:
!git clone https://github.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1.git speech2avatar

In [ ]:
# ============================================================================
# CONFIGURATION — EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
import os
from pathlib import Path
import torch

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

# Repository root (the folder that contains generator/, renderer/, app.py, etc.)
REPO_ROOT = Path("/workspace/speech2avatar/IMTalker")
if not REPO_ROOT.exists() and Path("/workspace/IMTalker").exists():
    REPO_ROOT = Path("/workspace/IMTalker")
PROJECT_ROOT = REPO_ROOT.parent

# Your emotion dataset root (flat layout: motion/ audio/ smirk/ gaze/)
EMOTION_DATASET_DIR = Path("/workspace/emotion_dataset")

# Checkpoints directory for base IMTalker models (renderer, wav2vec2, generator)
CKPT_DIR = REPO_ROOT / "checkpoints"
WAV2VEC_DIR = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT = CKPT_DIR / "renderer.ckpt"

# 1. Base generator checkpoint (the full 38.8M parameter model from cbsjtu01/IMTalker)
# MUST contain the core base weights (fmt.blocks.*, etc.)
BASE_GENERATOR_CKPT = CKPT_DIR / "generator.ckpt"

# 2. Style LoRA checkpoint (e.g. your live winner blink/avatar LoRA)
# If provided, its 46 LoRA layers are merged into the base generator weights BEFORE training emotion LoRA.
STYLE_LORA_CKPT = (
    PROJECT_ROOT / "checkpoints" / "live_winner" / "lora" / "ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
)
STYLE_LORA_RANK  = 64
STYLE_LORA_ALPHA = 128.0

# Legacy compatibility alias
PRETRAINED_GENERATOR_CKPT = BASE_GENERATOR_CKPT

# Where to store training outputs (logs, checkpoints, test videos)
EXP_ROOT = REPO_ROOT.parent / "emotion_exps"
EXP_NAME = "emotion_lora_v1"

# ---------------------------------------------------------------------------
# LoRA hyper-parameters (for Emotion Conditioning)
# ---------------------------------------------------------------------------
LORA_RANK    = 32      # LoRA rank for adaLN/decoder adapters
LORA_ALPHA   = 64.0    # LoRA alpha (scaling factor = alpha/rank = 2.0)
LORA_DROPOUT = 0.05    # Dropout on LoRA paths during training

# ---------------------------------------------------------------------------
# Training hyper-parameters
# ---------------------------------------------------------------------------
BATCH_SIZE   = 8        # Per-GPU batch size (use 4-8 for 24GB; 16 for 80GB A100)
LR           = 1e-4     # Learning rate for LoRA params; emotion_conditioner uses 2x LR
MAX_STEPS    = 100_000  # Total training steps
MAX_EPOCHS   = -1       # -1 = stop at MAX_STEPS; set e.g. 50 to stop by epoch count
SAVE_FREQ    = 5_000    # Save checkpoint every N steps
DISPLAY_FREQ = 2_000    # Validation every N steps
LOG_EVERY    = 50       # Text log write interval in steps
NUM_WORKERS  = 4        # DataLoader worker processes

# How many samples to hold out for validation (from the same flat dataset)
VAL_SPLIT    = 200      # Increase if you have more data; must be < total samples

# ---------------------------------------------------------------------------
# Resume / restart
# ---------------------------------------------------------------------------
AUTO_RESUME  = True     # Resume from last.ckpt if present
RESUME_CKPT  = None     # Or: "/path/to/specific.ckpt"

# ---------------------------------------------------------------------------
# Emotion filter (optional)
# Set to None to train on ALL 6 emotions:
#   0: neutral, 1: happy, 2: sad, 3: angry, 4: surprise, 5: fear
# Or restrict e.g. ["happy", "sad", "surprise"]
# ---------------------------------------------------------------------------
EMOTION_FILTER = None

# ---------------------------------------------------------------------------
# Per-epoch test video configuration
# ---------------------------------------------------------------------------
TEST_REF_IMAGE    = REPO_ROOT / "assets" / "3robert.jpeg" # Prioritize robot face avatar!
TEST_AUDIO        = '/workspace/speech2avatar/IMTalker/assets/audio_3.wav'    # None = auto-pick from assets/
TEST_EVERY_N      = 10       # Render a test video every N epochs
TEST_CROP         = False   # CRITICAL FOR ROBOT FACE: False = 512x512 resize without face_alignment crop!
TEST_EMOTION_ID   = 1       # Emotion shown in test video: 0=neutral, 1=happy, 2=sad, 3=angry, 4=surprise, 5=fear
TEST_CFG_SCALE    = 3.0
TEST_NFE          = 10
TEST_SEED         = 42

# ---------------------------------------------------------------------------
# Quick sanity prints
# ---------------------------------------------------------------------------
EXP_DIR = EXP_ROOT / EXP_NAME
print(f"repo:               {REPO_ROOT}")
print(f"project root:       {PROJECT_ROOT}")
print(f"emotion dataset:    {EMOTION_DATASET_DIR}")
print(f"base generator:     {BASE_GENERATOR_CKPT}")
print(f"style lora to merge:{STYLE_LORA_CKPT}")
print(f"renderer ckpt:      {RENDERER_CKPT}")
print(f"exp output:         {EXP_DIR}")
print(f"LoRA rank={LORA_RANK} alpha={LORA_ALPHA} | batch={BATCH_SIZE} lr={LR} max_steps={MAX_STEPS:,}")


In [ ]:
# Check and install required packages (safe to re-run)
import importlib.util, shutil, subprocess, sys

def _pip(*pkgs):
    print("[install]", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",
    "torchdiffeq":       "torchdiffeq==0.2.5",
    "einops":            "einops",
    "transformers":      "transformers>=4.38,<4.42",
    "librosa":           "librosa",
    "face_alignment":    "face_alignment==1.4.1",
    "av":                "av==12.0.0",
    "tensorboard":       "tensorboard",
    "timm":              "timm==1.0.9",
    "tqdm":              "tqdm",
    "huggingface_hub":   "huggingface_hub",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip(*missing)
else:
    print("All required packages already installed")

import torch
import pytorch_lightning as pl
print(f"torch:             {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {props.name}  {props.total_memory / 1e9:.1f} GB")
print(f"pytorch-lightning: {pl.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found — emotion LoRA training requires a GPU.")
assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (needed for test videos)"


In [ ]:
# ============================================================================
# Download all required pretrained models & checkpoints autonomously
# (Self-contained: does NOT depend on preprocessing notebooks 01/02)
# ============================================================================
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

CKPT_DIR.mkdir(parents=True, exist_ok=True)
if STYLE_LORA_CKPT:
    STYLE_LORA_CKPT.parent.mkdir(parents=True, exist_ok=True)

# 1. Download base IMTalker files (config.yaml, renderer.ckpt, generator.ckpt)
IMTALKER_FILES = ["config.yaml", "renderer.ckpt", "generator.ckpt"]
for fname in IMTALKER_FILES:
    dest = CKPT_DIR / fname
    alt_dest = REPO_ROOT.parent / "preprocessing_output" / "checkpoints" / fname
    if not dest.exists() and alt_dest.exists():
        dest = alt_dest
    if not dest.exists():
        print(f"[download] {fname} from cbsjtu01/IMTalker ...")
        hf_hub_download(repo_id="cbsjtu01/IMTalker", filename=fname, local_dir=str(CKPT_DIR))
        print(f"  ✓ {fname} ready: {dest}")
    else:
        print(f"[skip] {fname} already present: {dest}")

# 2. Download wav2vec2-base-960h model files
WAV2VEC_DIR.mkdir(parents=True, exist_ok=True)
W2V_FILES = [
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]
for rel_path in W2V_FILES:
    dest = CKPT_DIR / rel_path
    if not dest.exists():
        print(f"[download] {rel_path} from cbsjtu01/IMTalker ...")
        hf_hub_download(repo_id="cbsjtu01/IMTalker", filename=rel_path, local_dir=str(CKPT_DIR))
        print(f"  ✓ {dest.name} ready: {dest}")
    else:
        print(f"[skip] {dest.name} already present: {dest}")

# 3. Download style LoRA checkpoint (live winner LoRA) if configured and missing
if STYLE_LORA_CKPT and not STYLE_LORA_CKPT.exists():
    winner_rel = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
    if "ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt" in str(STYLE_LORA_CKPT):
        print(f"[download] Live-winner LoRA checkpoint from niloy629/hdtf_preprocess ...")
        try:
            download_root = PROJECT_ROOT / "checkpoints"
            download_root.mkdir(parents=True, exist_ok=True)
            dl_path = hf_hub_download(
                repo_id="niloy629/hdtf_preprocess",
                repo_type="dataset",
                filename=winner_rel,
                local_dir=str(download_root),
            )
            print(f"  ✓ Live-winner style LoRA checkpoint downloaded: {dl_path}")
        except Exception as e:
            print(f"[warn] Could not download style LoRA from niloy629/hdtf_preprocess: {e}")
            print(f"[info] Training will proceed using base generator.ckpt directly without style LoRA.")

# 4. Summary verification of all required model paths
print("\nModel checkpoints ready for training:")
print(f"  renderer.ckpt:        {RENDERER_CKPT} (exists={RENDERER_CKPT.exists()})")
print(f"  wav2vec2-base-960h:   {WAV2VEC_DIR} (exists={WAV2VEC_DIR.exists()})")
print(f"  base generator.ckpt:  {BASE_GENERATOR_CKPT} (exists={BASE_GENERATOR_CKPT.exists()})")
has_style = STYLE_LORA_CKPT.exists() if STYLE_LORA_CKPT else False
print(f"  style LoRA to merge:  {STYLE_LORA_CKPT} (exists={has_style})")

assert RENDERER_CKPT.exists(), f"renderer.ckpt missing at {RENDERER_CKPT}"
assert WAV2VEC_DIR.exists() and any(WAV2VEC_DIR.iterdir()), f"wav2vec2-base-960h missing at {WAV2VEC_DIR}"
assert BASE_GENERATOR_CKPT.exists(), f"Base generator.ckpt missing at {BASE_GENERATOR_CKPT}"
print("✓ All required model checkpoints verified successfully!")


In [ ]:
# Pre-flight checks: dataset, checkpoints, emotion code, test files
import cv2, sys, math, wave, struct
from pathlib import Path

# 1) Make sure the repo is importable
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

# 2) Check emotion conditioning code was added to the repo
try:
    from generator.FM import FMGenerator, EmotionConditioner
    print("✓ EmotionConditioner found in generator/FM.py")
except ImportError as e:
    raise ImportError(
        f"EmotionConditioner not found in generator/FM.py: {e}\n"
        "Make sure you ran the implementation step to add emotion conditioning."
    )

try:
    from generator.emotion_dataset import (
        EmotionAudioMotionDataset, EMOTION_MAP, EMOTION_NAMES, parse_emotion_from_stem
    )
    print("✓ generator/emotion_dataset.py found")
except ImportError as e:
    raise ImportError(
        f"generator/emotion_dataset.py not found: {e}\n"
        "Make sure emotion_dataset.py was created in generator/"
    )

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
assert training_script.exists(), (
    f"Missing {training_script}\n"
    "Make sure run_emotion_lora_training.py was created in generator/"
)
print(f"✓ {training_script.name} found")

from generator.train_lora import apply_emotion_lora_to_model
print("✓ apply_emotion_lora_to_model found in generator/train_lora.py")

# 3) Dataset directory structure check
for sub in ("motion", "audio", "smirk", "gaze"):
    d = EMOTION_DATASET_DIR / sub
    assert d.exists(), f"Missing {d} — prepare your emotion dataset in {EMOTION_DATASET_DIR}"
counts = {s: len(list((EMOTION_DATASET_DIR / s).glob("*.*")))
          for s in ("motion", "audio", "smirk", "gaze")}
print(f"\nDataset file counts: {counts}")
assert counts["motion"] >= max(10, VAL_SPLIT + 1), \
    f"Need at least {VAL_SPLIT + 10} clips, found {counts['motion']}"

# 4) Emotion label distribution check across all 6 emotions
motion_files = sorted((EMOTION_DATASET_DIR / "motion").glob("*.pt"))
from collections import Counter
emotion_distribution = Counter()
for mp in motion_files:
    eid = parse_emotion_from_stem(mp.stem)
    emotion_distribution[eid] += 1

print("\nEmotion distribution in dataset:")
for eid in range(6):
    cnt = emotion_distribution.get(eid, 0)
    bar = "█" * (cnt // 50) if cnt > 0 else "(missing)"
    print(f"  [{eid}] {EMOTION_NAMES[eid]:10s}: {cnt:5d}  {bar}")
assert sum(emotion_distribution.values()) > 0, "No valid emotion labels found in dataset"

# 5) Verify base generator has the full 108 weights (not a LoRA-only checkpoint)
base_raw = torch.load(str(BASE_GENERATOR_CKPT), map_location="cpu", weights_only=False)
base_dict = base_raw.get("ema_state_dict") or base_raw.get("state_dict", base_raw)
if isinstance(base_dict, dict) and "model" in base_dict and isinstance(base_dict["model"], dict):
    base_dict = base_dict["model"]
base_keys_count = len([k for k in base_dict.keys() if "fmt.blocks" in k])
assert base_keys_count > 50, \
    f"BASE_GENERATOR_CKPT ({BASE_GENERATOR_CKPT}) only has {base_keys_count} transformer keys! Must be full generator.ckpt."
print(f"\n✓ Base generator verified: {len(base_dict)} total keys ({base_keys_count} transformer blocks)")

# 6) Test reference image / audio (auto-pick or fallback)
if TEST_REF_IMAGE is None or not Path(TEST_REF_IMAGE).exists():
    for cand in [REPO_ROOT / "assets" / "3robert.jpeg", REPO_ROOT / "assets" / "source_3.jpg"]:
        if cand.exists():
            TEST_REF_IMAGE = cand
            break
    if TEST_REF_IMAGE is None:
        vids = sorted((REPO_ROOT.parent / "preprocessing_output" / "videos_512").glob("*.mp4"))
        if vids:
            cap = cv2.VideoCapture(str(vids[0]))
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
            ok, frm = cap.read(); cap.release()
            if ok:
                ref_out = EXP_DIR / "test_ref.png"
                ref_out.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(ref_out), frm)
                TEST_REF_IMAGE = ref_out

if TEST_AUDIO is None:
    for cand in [
        REPO_ROOT / "assets" / "audio_1.wav",
        PROJECT_ROOT / "personaplex" / "ai-thinking-sound.wav",
    ]:
        if cand.exists():
            TEST_AUDIO = cand
            break
    if TEST_AUDIO is None:
        synth_wav = EXP_DIR / "test_synth_audio.wav"
        synth_wav.parent.mkdir(parents=True, exist_ok=True)
        if not synth_wav.exists():
            with wave.open(str(synth_wav), "w") as wf:
                wf.setnchannels(1)
                wf.setsampwidth(2)
                wf.setframerate(16000)
                samples = [int(32767 * 0.3 * math.sin(2 * math.pi * 440 * i / 16000)) for i in range(16000 * 3)]
                wf.writeframes(struct.pack(f"<{len(samples)}h", *samples))
        TEST_AUDIO = synth_wav

strategy = "ddp_find_unused_parameters_true" if n_gpu > 1 else "auto"
eff_batch = BATCH_SIZE * n_gpu
print(f"\n✓ gpus={n_gpu}  strategy={strategy}  effective batch={eff_batch}")
if TEST_REF_IMAGE:
    print(f"✓ test image: {TEST_REF_IMAGE}")
if TEST_AUDIO:
    print(f"✓ test audio: {TEST_AUDIO}")
print(f"✓ test crop:  {TEST_CROP} (False is safe for robot faces)")
print(f"✓ test emotion ID: {TEST_EMOTION_ID} ({EMOTION_NAMES.get(TEST_EMOTION_ID, 'unknown')})")
print("\nAll pre-flight checks passed ✓")


## Launch Training

**Option A (recommended — required for multi-GPU)**: Launches training as a
background process using the repository's exact DDP configuration. It survives
closing this notebook or disconnecting from the pod, and streams output to
`train_console.log` + `train_log.txt`.

**Option B (single GPU only)**: Trains directly in the notebook (with live
progress bar; blocks this cell until complete).


In [ ]:
# OPTION A — Launch emotion LoRA training as a background process.
# Required for multi-GPU; survives kernel restarts.
import os, subprocess, sys

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
console_log = EXP_DIR / "train_console.log"
EXP_DIR.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(training_script),
    "--repo_root",            str(REPO_ROOT),
    "--dataset_path",         str(EMOTION_DATASET_DIR),
    "--base_generator_ckpt",   str(BASE_GENERATOR_CKPT),
    "--exp_root",             str(EXP_ROOT),
    "--exp_name",             EXP_NAME,
    "--wav2vec_model_path",   str(WAV2VEC_DIR),
    "--renderer_ckpt",        str(RENDERER_CKPT),
    "--lora_rank",            str(LORA_RANK),
    "--lora_alpha",           str(LORA_ALPHA),
    "--lora_dropout",         str(LORA_DROPOUT),
    "--batch_size",           str(BATCH_SIZE),
    "--lr",                   str(LR),
    "--max_steps",            str(MAX_STEPS),
    "--max_epochs",           str(MAX_EPOCHS),
    "--save_freq",            str(SAVE_FREQ),
    "--display_freq",         str(DISPLAY_FREQ),
    "--log_every_n_steps",    str(LOG_EVERY),
    "--num_workers",          str(NUM_WORKERS),
    "--val_split",            str(VAL_SPLIT),
    "--test_every_n_epochs",  str(TEST_EVERY_N),
    "--test_crop",            "1" if TEST_CROP else "0",
    "--test_emotion_id",      str(TEST_EMOTION_ID),
    "--test_cfg",             str(TEST_CFG_SCALE),
    "--test_nfe",             str(TEST_NFE),
    "--test_seed",            str(TEST_SEED),
]
if STYLE_LORA_CKPT and STYLE_LORA_CKPT.exists():
    cmd += [
        "--style_lora_ckpt",  str(STYLE_LORA_CKPT),
        "--style_lora_rank",  str(STYLE_LORA_RANK),
        "--style_lora_alpha", str(STYLE_LORA_ALPHA),
    ]
if TEST_REF_IMAGE:
    cmd += ["--test_ref", str(TEST_REF_IMAGE)]
if TEST_AUDIO:
    cmd += ["--test_aud", str(TEST_AUDIO)]
if TEST_EVERY_N:
    cmd += ["--test_out_dir", str(EXP_DIR / "test_videos")]
if not AUTO_RESUME:
    cmd.append("--no_auto_resume")
if RESUME_CKPT:
    cmd += ["--resume_ckpt", str(RESUME_CKPT)]
if EMOTION_FILTER:
    cmd += ["--emotion_filter"] + list(EMOTION_FILTER)

print("Launching:", " ".join(cmd[:10]), "...\n")
with open(console_log, "a") as logf:
    kwargs = dict(stdout=logf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT))
    if os.name == "posix":
        kwargs["start_new_session"] = True
    else:
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    proc = subprocess.Popen(cmd, **kwargs)

(EXP_DIR / "train.pid").write_text(str(proc.pid))
print(f"Training started  | pid={proc.pid}")
print(f"Console log:      {console_log}")
print(f"Checkpoint dir:   {EXP_DIR / 'checkpoints'}")
print("Run the MONITOR cell below to follow progress.")


In [ ]:
# OPTION B — Train directly in this notebook (single GPU only).
# This cell blocks until training finishes. For multi-GPU, use OPTION A.
assert torch.cuda.device_count() <= 1, \
    f"{torch.cuda.device_count()} GPUs detected — use OPTION A for multi-GPU training."

import sys
for p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if p not in sys.path:
        sys.path.insert(0, p)

import argparse
import importlib.util

spec = importlib.util.spec_from_file_location(
    "run_emotion_lora_training",
    str(REPO_ROOT / "generator" / "run_emotion_lora_training.py")
)
relt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(relt)

args = argparse.Namespace(
    repo_root=str(REPO_ROOT),
    dataset_path=str(EMOTION_DATASET_DIR),
    base_generator_ckpt=str(BASE_GENERATOR_CKPT),
    style_lora_ckpt=str(STYLE_LORA_CKPT) if (STYLE_LORA_CKPT and STYLE_LORA_CKPT.exists()) else None,
    style_lora_rank=STYLE_LORA_RANK,
    style_lora_alpha=STYLE_LORA_ALPHA,
    pretrained_ckpt=None,
    exp_root=str(EXP_ROOT),
    exp_name=EXP_NAME,
    wav2vec_model_path=str(WAV2VEC_DIR),
    renderer_ckpt=str(RENDERER_CKPT),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_steps=MAX_STEPS,
    max_epochs=MAX_EPOCHS,
    save_freq=SAVE_FREQ,
    display_freq=DISPLAY_FREQ,
    log_every_n_steps=LOG_EVERY,
    num_workers=NUM_WORKERS,
    val_split=VAL_SPLIT,
    resume_ckpt=str(RESUME_CKPT) if RESUME_CKPT else None,
    no_auto_resume=not AUTO_RESUME,
    test_ref=str(TEST_REF_IMAGE) if TEST_REF_IMAGE else None,
    test_aud=str(TEST_AUDIO) if TEST_AUDIO else None,
    test_out_dir=str(EXP_DIR / "test_videos"),
    test_every_n_epochs=TEST_EVERY_N,
    test_crop=1 if TEST_CROP else 0,
    test_emotion_id=TEST_EMOTION_ID,
    test_cfg=TEST_CFG_SCALE,
    test_nfe=TEST_NFE,
    test_seed=TEST_SEED,
    emotion_filter=EMOTION_FILTER,
    list_path=None,
    val_list_path=None,
)
relt.main(args)


In [ ]:
# MONITOR — re-run this cell anytime to check training progress
import os
from pathlib import Path

def _tail(path, n=20):
    path = Path(path)
    if not path.exists():
        print(f"  (no {path.name} yet)")
        return
    lines = path.read_text(errors="replace").splitlines()
    print(f"--- Last {min(n, len(lines))} lines of {path.name} ---")
    print("\n".join(lines[-n:]))

pid_file = EXP_DIR / "train.pid"
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f"Process {pid} is RUNNING ✓")
    except OSError:
        print(f"Process {pid} is NOT running (finished or stopped)")

ckpts = sorted((EXP_DIR / "checkpoints").glob("*.ckpt"))
print(f"Checkpoints saved: {len(ckpts)}")
for c in ckpts[-3:]:
    print(f"  {c.name} ({c.stat().st_size / 1e6:.1f} MB)")

vids = sorted((EXP_DIR / "test_videos").glob("*.mp4"))
print(f"Test videos generated: {len(vids)}")
for v in vids[-3:]:
    print(f"  {v.name} ({v.stat().st_size / 1e6:.1f} MB)")

print()
_tail(EXP_DIR / "train_log.txt", 15)
print()
_tail(EXP_DIR / "train_console.log", 15)


In [ ]:
# STOP training started with OPTION A (sends SIGTERM; last checkpoint saved)
STOP_TRAINING = False

if not STOP_TRAINING:
    print("Set STOP_TRAINING = True and re-run this cell to stop background training.")
else:
    import os, signal
    pid_file = EXP_DIR / "train.pid"
    assert pid_file.exists(), "No train.pid found — was training started with Option A?"
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to process {pid}. Last checkpoint preserved.")
    except ProcessLookupError:
        print(f"Process {pid} already stopped.")


## Validate the Trained Checkpoint

Run the cells below **after** training finishes to verify:
1. The checkpoint is loadable and contains the emotion conditioning weights
2. The emotion projection layer has non-zero weights
3. Different emotion IDs produce distinct conditioning vectors


In [ ]:
# Load the trained emotion LoRA checkpoint and verify its contents
import torch, sys
from pathlib import Path

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

EVAL_CKPT = None  # None = auto-pick last.ckpt

if EVAL_CKPT is None:
    last_ckpt = EXP_DIR / "checkpoints" / "last.ckpt"
    step_ckpts = sorted((EXP_DIR / "checkpoints").glob("step=*.ckpt"),
                        key=lambda p: p.stat().st_mtime)
    EVAL_CKPT = last_ckpt if last_ckpt.exists() else (step_ckpts[-1] if step_ckpts else None)

assert EVAL_CKPT and Path(EVAL_CKPT).exists(), \
    f"No checkpoint found in {EXP_DIR / 'checkpoints'}"
print(f"Evaluating: {EVAL_CKPT}")

ckpt = torch.load(str(EVAL_CKPT), map_location="cpu", weights_only=False)
state = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt)
if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
    state = state["model"]
state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}

# Check emotion-specific keys
emotion_keys = [k for k in state if "emotion" in k]
lora_keys    = [k for k in state if "lora_"  in k]
print(f"\nEmotion conditioner keys: {len(emotion_keys)}")
for k in emotion_keys:
    print(f"  {k}: {tuple(state[k].shape)}")
print(f"\nLoRA adapter keys: {len(lora_keys)}")
print(f"Total keys in checkpoint: {len(state)}")

proj_w = state.get("emotion_conditioner.emotion_projection.0.weight")
if proj_w is not None:
    norm = proj_w.norm().item()
    print(f"\nemotion_projection weight norm: {norm:.6f}")
    if norm < 1e-6:
        print("  WARNING: weights still zero — training may not have converged yet")
    else:
        print("  ✓ Emotion projection learned (norm > 0)")


In [ ]:
# Verify the emotion conditioner produces distinct, non-zero biases per emotion
import torch, sys
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "generator"))

import argparse
from generator.options.base_options import BaseOptions
from generator.FM import FMGenerator, EmotionConditioner, EMOTION_MAP, EMOTION_NAMES
from generator.train_lora import apply_emotion_lora_to_model

# Build model with same opt as training
parser = argparse.ArgumentParser()
parser = BaseOptions().initialize(parser)
parser.add_argument("--skip_fm_audio_encoder", action="store_true")
opt = parser.parse_args([])
opt.rank = "cpu"
opt.wav2vec_model_path = str(WAV2VEC_DIR)
opt.skip_fm_audio_encoder = True

model = FMGenerator(opt)
apply_emotion_lora_to_model(model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=0.0)
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Model loaded | missing={len(missing)} unexpected={len(unexpected)}")

# Check canonical 6 emotion embeddings
print("\nEmotion bias norms (T=50 frames):")
biases = {}
for idx in range(6):
    name = EMOTION_NAMES[idx]
    eid = torch.tensor([idx], dtype=torch.long)
    with torch.no_grad():
        bias = model.emotion_conditioner(eid, T=50, train=False)  # [1, 50, 32]
    norm = bias.mean(dim=1).norm().item()
    biases[name] = bias.squeeze(0).mean(0)  # [32]
    print(f"  [{idx}] {name:10s}: norm={norm:.6f}")

# Pairwise distances
print("\nPairwise L2 distances between emotion embeddings:")
names = [EMOTION_NAMES[i] for i in range(6)]
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if j <= i:
            continue
        dist = (biases[n1] - biases[n2]).norm().item()
        print(f"  {n1:10s} vs {n2:10s}: {dist:.4f}")


## After Training — Output Files

All outputs are in `{EXP_ROOT}/{EXP_NAME}/`:

| Path | Content |
|------|---------|
| `checkpoints/last.ckpt` | Latest checkpoint (auto-resume source) |
| `checkpoints/step=XXXXXX.ckpt` | Periodic checkpoints (every `SAVE_FREQ` steps) |
| `train_log.txt` | Human-readable epoch/step/loss/lr log |
| `train_console.log` | Full console output (Option A) |
| `version_*/metrics.csv` | TensorBoard + CSV metrics |
| `test_videos/epoch_XXXX.mp4` | Per-epoch test videos showing the avatar in action |

## Using the Trained Checkpoint in Your Live System

Pass these arguments to your live server (in addition to your existing args):

```bash
# In run_live.sh or start_winner_live.sh, set:
EMOTION_LORA_PATH="/path/to/emotion_exps/emotion_lora_v1/checkpoints/last.ckpt"
EMOTION_LORA_RANK=32
EMOTION_LORA_ALPHA=64.0
EMOTION_ID=1   # 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear
```

Or deploy with the companion notebook `06_emotion_live_pipeline.ipynb` which
loads this checkpoint and lets you configure the emotion at startup or during conversations.
